# Módulo 10: Un catálogo gobernado de agentes con AWS Agent Registry (30 min)

En los Módulos 6 y 9 el orquestador conocía a sus especialistas **cuando escribías el código**: los importabas o escribías su URL. Con tres agentes funciona; con los cientos de herramientas, servidores MCP y agentes que acaban publicando los equipos de una empresa, no. Además surgen preguntas de gobierno: ¿quién es el responsable de esta herramienta?, ¿alguien la ha revisado?, ¿maneja datos personales?

En este módulo usarás **AWS Agent Registry**, un catálogo privado y gobernado donde los equipos **publican** sus capacidades, un curador las **aprueba** y los agentes las **descubren**. Lo conectarás con Strands de dos formas:

1. **El registro como servidor MCP** de un agente de Strands, para descubrir capacidades mientras desarrollas.
2. **Descubrimiento en tiempo de ejecución:** un orquestador sin herramientas propias que busca en el registro, valida, conecta y ejecuta lo que necesita en cada petición.

Basado en la documentación oficial de [AWS Agent Registry](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry.html), el artículo [Discover tools at runtime with AWS Agent Registry](https://aws.amazon.com/blogs/machine-learning/discover-tools-at-runtime-with-aws-agent-registry/) y las guías de Strands sobre [herramientas MCP](https://strandsagents.com/docs/user-guide/sdk/tools/mcp-tools/), [A2A](https://strandsagents.com/docs/user-guide/sdk/multi-agent/agent-to-agent/) y [skills](https://strandsagents.com/docs/user-guide/sdk/plugins/skills/).

**Prerrequisitos:** Módulos 3 (skills) y 9 (A2A). Credenciales de AWS en una región con AWS Agent Registry y permisos para crear y borrar registros (`agent-registry:*`). Strands usa Amazon Bedrock por defecto. Ejecuta `uv sync` en la raíz del proyecto: instala una versión de boto3 que ya incluye los clientes `agent-registry-control` y `agent-registry`.

**Archivos de esta carpeta:**

| Archivo | Para qué sirve |
|---|---|
| `servidor_mcp_pedidos.py` | Servidor MCP del equipo de Pedidos (`consultar_pedido` y `consultar_envio`) |
| `herramientas_soporte.py` | El especialista de soporte técnico del Módulo 9 |
| `servidor_soporte_tecnico.py` | Expone a ese especialista como servicio A2A |
| `skills/politica-devoluciones/SKILL.md` | Skill del equipo de Atención al cliente |

> 💰 **Recursos en tu cuenta:** el notebook crea un registro con un nombre único y lo borra en la Parte 8. Si te saltas partes o algo falla, ejecuta igualmente la celda de limpieza.

---

## Contexto: ¿qué es AWS Agent Registry?

AWS Agent Registry es un servicio gestionado de **catálogo y descubrimiento** para servidores MCP, agentes, *skills* y recursos personalizados. Se organiza en dos recursos:

- **Registro** (*registry*): el catálogo. Tiene su propia autorización (IAM o JWT), su configuración de aprobación y, opcionalmente, un esquema de metadatos obligatorios.
- **Entrada** (*registry record*): cada recurso publicado. Contiene metadatos que se pueden buscar (nombre, descripción, versión, metadatos personalizados) y un **descriptor** con la información del protocolo.

> En español usamos *registro* para el catálogo y *entrada* para cada recurso publicado, para no confundir *registry* con *record*.

| Tipo de entrada (`recordType`) | Descriptor | Qué guarda |
|---|---|---|
| `MCP` | `mcpServer` | El `server.json` del servidor (incluida la URL de conexión) y la definición de sus herramientas |
| `AGENT` | `a2aAgentCard` | La Agent Card A2A del agente (versión de esquema 0.3) |
| `SKILL` | `agentSkillsDefinition` | El contenido del `SKILL.md` y, opcionalmente, paquetes o repositorio |
| `CUSTOM` | `custom` | Cualquier JSON válido (APIs, funciones Lambda, bases de conocimiento…) |

El registro **valida** los descriptores MCP y A2A contra el esquema oficial de cada protocolo.

### Cuatro roles

| Rol | Qué hace | API |
|---|---|---|
| **Administrador** | Crea el registro: autorización, aprobación y esquema de metadatos | `agent-registry-control` · `CreateRegistry` |
| **Publicador** | Describe su recurso y lo envía a revisión | `CreateRegistryRecord`, `SubmitRegistryRecordForApproval` |
| **Curador** | Aprueba, rechaza o retira entradas | `UpdateRegistryRecordStatus` |
| **Consumidor** | Busca y usa las entradas aprobadas, por API o por MCP | `agent-registry` · `SearchDiscoverableRegistryRecords` |

Hay dos clientes de boto3: el **plano de control** (`agent-registry-control`) gestiona registros y entradas en cualquier estado; el **plano de datos** (`agent-registry`) solo ve las entradas aprobadas.

### Ciclo de vida de una entrada

```
CREATING ─▶ DRAFT ─▶ PENDING_APPROVAL ─┬─▶ APPROVED ─▶ DEPRECATED (final)
              ▲                        └─▶ REJECTED
              └─── el publicador la edita y vuelve a enviarla a revisión
```

**Solo las entradas `APPROVED` son descubribles.** Esa es la base del gobierno: nada llega a los agentes sin pasar por el curador.

### El caso de hoy: el catálogo de TechStore

Tres equipos de TechStore quieren compartir sus capacidades y un cuarto intenta publicar algo que no debería:

| Equipo | Publica | Tipo |
|---|---|---|
| Pedidos | Servidor MCP para consultar pedidos y envíos | `MCP` |
| Soporte | El especialista técnico del Módulo 9 | `AGENT` (A2A) |
| Atención al cliente | La política de devoluciones | `SKILL` |
| Marketing | Un exportador de la base de clientes | `MCP` 🚩 |

Vas a recorrer el ciclo completo, asumiendo cada rol:

| Parte | Rol | Qué aprendes |
|---|---|---|
| 0 | — | Preparación: clientes de boto3 y servicios locales |
| 1 | Administrador | Crear un registro con aprobación manual y metadatos obligatorios |
| 2 | Publicador | Publicar un servidor MCP, un agente A2A y una skill |
| 3 | Curador | Revisar, aprobar y rechazar |
| 4 | Consumidor | Buscar con lenguaje natural y con filtros |
| 5 | Desarrollador | El registro como servidor MCP de un agente de Strands |
| 6 | Orquestador | Descubrir → validar → instanciar → ejecutar |
| 7 | Gobierno | Cambios que pasan por revisión y retirada de capacidades |
| 8 | — | Limpieza |

---

## Parte 0: Preparación

Creamos los dos clientes de boto3 y un sufijo aleatorio para que el nombre de tu registro no choque con el de otros participantes. Definimos también tres funciones auxiliares: dos que esperan a que terminen las operaciones asíncronas del registro y otra que muestra todas sus entradas.

In [ ]:
import json
import time
import uuid

import boto3
from botocore.exceptions import ClientError
from IPython.display import Markdown, display

sesion = boto3.session.Session()
REGION = sesion.region_name

# Regiones de AWS Agent Registry en su lanzamiento
REGIONES_REGISTRY = {"us-east-1", "us-west-2", "eu-west-1", "ap-northeast-1", "ap-southeast-2"}
if REGION not in REGIONES_REGISTRY:
    print(f"⚠️ AWS Agent Registry podría no estar disponible en {REGION}. Regiones: {sorted(REGIONES_REGISTRY)}")

control = sesion.client("agent-registry-control")  # plano de control: registros y entradas en cualquier estado
descubrimiento = sesion.client("agent-registry")   # plano de datos: solo entradas aprobadas

SUFIJO = uuid.uuid4().hex[:6]
print(f"Región: {REGION} | Sufijo: {SUFIJO}")


def esperar_registro(id_registro: str) -> str:
    """Espera a que el registro termine de crearse y devuelve su estado."""
    while (estado := control.get_registry(registryId=id_registro)["status"]) == "CREATING":
        time.sleep(5)
    return estado


def esperar_entrada(id_entrada: str) -> str:
    """Espera a que termine un cambio asíncrono en una entrada y devuelve su estado."""
    while (estado := control.get_registry_record(registryId=ID_REGISTRO, recordId=id_entrada)["status"]) in (
        "CREATING",
        "UPDATING",
    ):
        time.sleep(2)
    return estado


def tabla_entradas() -> None:
    """Muestra todas las entradas del registro. El plano de control ve también las no aprobadas."""
    filas = ["| Entrada | Tipo | Estado | Metadatos |", "|---|---|---|---|"]
    lista = control.list_registry_records(registryId=ID_REGISTRO)["registryRecords"]
    for e in sorted(lista, key=lambda e: e["name"]):
        cumplimiento = e.get("customMetadataSchemaComplianceStatus", "-")
        filas.append(f"| `{e['name']}` | {e['recordType']} | **{e['status']}** | {cumplimiento} |")
    display(Markdown("\n".join(filas)))

### Los servicios de los equipos

El registro **no ejecuta nada**: solo guarda la descripción de cada recurso y cómo conectarse a él. Los servicios tienen que estar en marcha en algún sitio. En producción vivirían en Amazon Bedrock AgentCore Runtime, AgentCore Gateway o cualquier otra plataforma; aquí los lanzamos en segundo plano desde el notebook, como en el Módulo 9:

- El **servidor MCP de Pedidos** (`servidor_mcp_pedidos.py`), con transporte *streamable HTTP*.
- El **agente A2A de Soporte** (`servidor_soporte_tecnico.py`).

Pedimos al sistema puertos libres y esperamos a que ambos respondan. Sus logs se guardan en `servidor_mcp.log` y `servidor_a2a.log`.

In [ ]:
import socket
import subprocess
import sys

import httpx


def puerto_libre() -> int:
    """Pide al sistema operativo un puerto TCP libre."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def arrancar_servicio(script: str, puerto: int, url_salud: str, log: str):
    """Lanza un servicio en segundo plano y espera a que responda en url_salud."""
    salida = open(log, "w")
    proceso = subprocess.Popen([sys.executable, script, str(puerto)], stdout=salida, stderr=subprocess.STDOUT)
    for _ in range(30):
        if proceso.poll() is not None:
            raise RuntimeError(f"{script} se detuvo. Revisa {log}")
        try:
            httpx.get(url_salud, timeout=2)
            return proceso, salida
        except httpx.HTTPError:
            time.sleep(1)
    raise RuntimeError(f"{script} no respondió a tiempo. Revisa {log}")


PUERTO_MCP, PUERTO_A2A = puerto_libre(), puerto_libre()
URL_MCP_PEDIDOS = f"http://127.0.0.1:{PUERTO_MCP}/mcp"
URL_SOPORTE = f"http://127.0.0.1:{PUERTO_A2A}"

servicio_mcp, log_mcp = arrancar_servicio("servidor_mcp_pedidos.py", PUERTO_MCP, URL_MCP_PEDIDOS, "servidor_mcp.log")
servicio_a2a, log_a2a = arrancar_servicio(
    "servidor_soporte_tecnico.py", PUERTO_A2A, f"{URL_SOPORTE}/.well-known/agent-card.json", "servidor_a2a.log"
)
print(f"✅ Servidor MCP de Pedidos: {URL_MCP_PEDIDOS}")
print(f"✅ Agente A2A de Soporte:   {URL_SOPORTE}")

---

## Parte 1: El administrador crea el registro

Al crear el registro decides tres cosas:

| Opción | Qué eliges aquí | Alternativas |
|---|---|---|
| **Autorización** (`discoveryConfiguration`) | IAM (`AWS_IAM`, el valor por defecto): se busca con credenciales de AWS firmadas con SigV4 | `CUSTOM_JWT`: tokens de tu proveedor de identidad (por ejemplo, Amazon Cognito). No se puede cambiar después |
| **Aprobación** (`approvalConfiguration`) | Sin reglas de autoaprobación (`autoApprovalRules: []`): un curador revisa cada entrada | Autoaprobación: las entradas enviadas pasan directamente a `APPROVED` |
| **Metadatos personalizados** (`customMetadataSchemaConfiguration`) | Un esquema que **toda** entrada debe cumplir | Campos adicionales por tipo de entrada (`recordTypeSchemaOverrides`) |

El esquema de metadatos convierte las reglas de gobierno en campos con tipo. En TechStore, toda entrada debe declarar su **equipo** responsable y la **clasificación** de los datos que maneja; también puede indicar si **contiene datos personales** (PII). Cada campo puede ser texto, una lista de valores permitidos (`enum`), una URL o un booleano.

La creación tarda alrededor de un minuto.

In [ ]:
ESQUEMA_METADATOS = {
    "type": "object",
    "properties": {
        "equipo": {"type": "string"},
        "clasificacion_datos": {"type": "string", "enum": ["publico", "interno", "confidencial"]},
        "contiene_pii": {"type": "boolean"},
    },
    "required": ["equipo", "clasificacion_datos"],
}

respuesta = control.create_registry(
    name=f"techstore-catalogo-{SUFIJO}",
    description="Catálogo gobernado de agentes, servidores MCP y skills de TechStore (Módulo 10)",
    approvalConfiguration={"autoApprovalRules": []},  # sin autoaprobación: revisa un curador
    customMetadataSchemaConfiguration={"defaultSchema": json.dumps(ESQUEMA_METADATOS)},
    tags={"workshop": "modulo-10"},
)
ARN_REGISTRO = respuesta["registryArn"]
ID_REGISTRO = ARN_REGISTRO.split("/")[-1]

print(f"Creando {ARN_REGISTRO} ...")
print(f"Estado: {esperar_registro(ID_REGISTRO)}")

configuracion = control.get_registry(registryId=ID_REGISTRO)
print(f"Autorización: {configuracion['discoveryConfiguration']['authorizerType']}")
print(f"Reglas de autoaprobación: {configuracion['approvalConfiguration']['autoApprovalRules']}")

---

## Parte 2: Los equipos publican sus capacidades

Una entrada tiene estos campos:

| Campo | Para qué sirve |
|---|---|
| `name`, `displayName` | Nombre único en el registro y nombre legible |
| `description` | **Lo que indexa la búsqueda.** Escríbela con las palabras que usaría quien la busca, no con jerga interna |
| `recordType`, `descriptors` | El tipo de entrada y su descriptor de protocolo |
| `recordVersion` | La versión del recurso (`name` + `recordVersion` debe ser único) |
| `customMetadata` | Los campos del esquema de la Parte 1 |

La función `publicar` crea la entrada, espera a que pase de `CREATING` a `DRAFT` y guarda su ID.

In [ ]:
entradas = {}  # nombre de la entrada → recordId


def publicar(nombre: str, **campos) -> str:
    """Crea una entrada en el registro, espera a que quede en DRAFT y devuelve su ID."""
    respuesta = control.create_registry_record(registryId=ID_REGISTRO, name=nombre, **campos)
    id_entrada = respuesta["recordArn"].split("/")[-1]
    entradas[nombre] = id_entrada
    print(f"📝 {nombre}: {respuesta['status']} → {esperar_entrada(id_entrada)}")
    return id_entrada

### 2.1 Un servidor MCP

El descriptor MCP tiene dos partes:

- **El servidor:** un [`server.json`](https://github.com/modelcontextprotocol/registry) del registro oficial de MCP con el nombre (`espacio/nombre`), la descripción, la versión y, en `remotes`, **cómo conectarse**.
- **Las herramientas:** sus definiciones con nombre, descripción y esquema de entrada.

En una empresa, el *pipeline* de CI/CD del equipo generaría el descriptor a partir del servidor real. Hacemos lo mismo: nos conectamos con el `MCPClient` de Strands, leemos las herramientas y las publicamos.

In [ ]:
from strands.tools.mcp import MCPClient

# El "pipeline de CI" del equipo de Pedidos: lee las herramientas del servidor real
with MCPClient(url=URL_MCP_PEDIDOS) as cliente:
    herramientas_mcp = [
        h.mcp_tool.model_dump(mode="json", by_alias=True, exclude_none=True) for h in cliente.list_tools_sync()
    ]

servidor_json = {
    "name": "io.techstore/pedidos",
    "description": "Consulta pedidos y envíos de TechStore",
    "version": "1.0.0",
    "remotes": [{"type": "streamable-http", "url": URL_MCP_PEDIDOS}],
}

publicar(
    "techstore-pedidos",
    displayName="Pedidos de TechStore",
    description="Servidor MCP del equipo de Pedidos. Consulta un pedido (producto, importe, estado y fecha de "
    "entrega) y el envío de un paquete (transportista, número de seguimiento, dónde está y cuándo llega).",
    recordType="MCP",
    recordVersion="1.0.0",
    descriptors={
        "mcpServer": {
            "data": json.dumps(servidor_json),
            "dataSchemaVersion": "2025-12-11",
            "additionalData": {
                "tools": {"data": json.dumps({"tools": herramientas_mcp}), "dataSchemaVersion": "2025-11-25"}
            },
        }
    },
    customMetadata={"equipo": "pedidos", "clasificacion_datos": "interno", "contiene_pii": False},
)
print("Herramientas publicadas:", [h["name"] for h in herramientas_mcp])

### 2.2 Un agente A2A

Para un agente A2A, el descriptor es su **Agent Card**. El servidor A2A de Strands la publica en `/.well-known/agent-card.json`, así que el equipo de Soporte solo tiene que descargarla y registrarla tal cual.

In [ ]:
tarjeta = httpx.get(f"{URL_SOPORTE}/.well-known/agent-card.json").json()

publicar(
    "techstore-soporte-tecnico",
    displayName="Soporte técnico de TechStore",
    description="Agente A2A del equipo de Soporte. Diagnostica problemas y averías de dispositivos electrónicos "
    "(auriculares que no se emparejan, teclados, hubs USB-C) y propone soluciones paso a paso.",
    recordType="AGENT",
    recordVersion="1.0.0",
    descriptors={"a2aAgentCard": {"data": json.dumps(tarjeta), "dataSchemaVersion": "0.3"}},
    customMetadata={"equipo": "soporte", "clasificacion_datos": "interno", "contiene_pii": False},
)
print(f"Agent Card: {tarjeta['name']} | {tarjeta['url']} | skills: {[s['id'] for s in tarjeta['skills']]}")

### 2.3 Una skill

Una skill es un paquete de instrucciones que un agente activa cuando las necesita (Módulo 3). El registro guarda el contenido del `SKILL.md`; así, cualquier agente de la empresa puede descubrir la política de devoluciones **sin copiar el archivo** a su repositorio.

In [ ]:
from pathlib import Path

skill_md = Path("skills/politica-devoluciones/SKILL.md").read_text()

publicar(
    "politica-devoluciones",
    displayName="Política de devoluciones",
    description="Skill del equipo de Atención al cliente. Decide si un pedido admite devolución, garantía o "
    "reembolso del dinero, y explica al cliente los pasos y los plazos.",
    recordType="SKILL",
    recordVersion="1.0.0",
    descriptors={"agentSkillsDefinition": {"additionalData": {"skillMd": {"data": skill_md}}}},
    customMetadata={"equipo": "atencion-cliente", "clasificacion_datos": "publico", "contiene_pii": False},
)
print(skill_md.split("---")[1].strip())

### 2.4 Dos publicaciones problemáticas

No todo lo que se publica debería llegar a los agentes:

- **a)** El equipo de Pedidos sube una beta y **olvida los metadatos** obligatorios.
- **b)** Marketing publica un servidor que **exporta datos personales** de los clientes. Sus metadatos están completos y son sinceros: declara `confidencial` y `contiene_pii`.

In [ ]:
# a) Una entrada sin los metadatos obligatorios
publicar(
    "pedidos-beta",
    description="Versión beta del servidor de Pedidos.",
    recordType="MCP",
    recordVersion="0.9.0",
    descriptors={
        "mcpServer": {
            "data": json.dumps({**servidor_json, "name": "io.techstore/pedidos-beta", "version": "0.9.0"}),
            "dataSchemaVersion": "2025-12-11",
        }
    },
)

# b) Un servidor que expone datos personales de los clientes
exportador_json = {
    "name": "io.techstore/exportador-clientes",
    "description": "Exporta la base de clientes a CSV",
    "version": "0.1.0",
    "remotes": [{"type": "streamable-http", "url": "https://exportador.marketing.example.com/mcp"}],
}
publicar(
    "exportador-clientes",
    displayName="Exportador de clientes",
    description="Servidor MCP del equipo de Marketing. Exporta la base de datos completa de clientes "
    "(nombre, email, dirección y número de tarjeta) a un fichero CSV.",
    recordType="MCP",
    recordVersion="0.1.0",
    descriptors={"mcpServer": {"data": json.dumps(exportador_json), "dataSchemaVersion": "2025-12-11"}},
    customMetadata={"equipo": "marketing", "clasificacion_datos": "confidencial", "contiene_pii": True},
)

tabla_entradas()

Todas las entradas están en `DRAFT`, pero `pedidos-beta` aparece como **`NON_COMPLIANT`**: el registro la aceptó como borrador y anotó que no cumple el esquema. Ahora los publicadores las envían a revisión:

In [ ]:
for nombre, id_entrada in entradas.items():
    try:
        estado = control.submit_registry_record_for_approval(registryId=ID_REGISTRO, recordId=id_entrada)["status"]
        print(f"📨 {nombre}: {estado}")
    except ClientError as error:
        print(f"⛔ {nombre}: {error.response['Error']['Code']}: {error.response['Error']['Message']}")

Este es el **primer control de gobierno**, y es automático: una entrada sin los metadatos obligatorios **no puede ni enviarse a revisión**. Las demás pasan a `PENDING_APPROVAL` y esperan a un curador, porque el registro no tiene reglas de autoaprobación.

Fíjate en que el exportador de Marketing **sí** ha pasado: sus metadatos son válidos. Decidir si algo que maneja datos personales debe publicarse no es una cuestión de esquema, sino de política. Esa decisión le corresponde al curador.

---

## Parte 3: El curador revisa

Antes de nada, comprobemos qué ven los consumidores ahora mismo. El plano de datos solo devuelve entradas aprobadas, y todavía no hay ninguna:

In [ ]:
resultado = descubrimiento.search_discoverable_registry_records(
    registryIds=[ARN_REGISTRO], searchQuery="pedidos de TechStore"
)
print("Resultados antes de aprobar:", resultado["registryRecords"])

Ahora el curador revisa la cola de entradas pendientes. Escribimos la política de TechStore como una función de Python: rechaza todo lo que exponga PII o datos confidenciales, o cuya descripción sea demasiado pobre para que otros equipos la encuentren.

> 💡 En producción, el registro puede enviar un evento a **Amazon EventBridge** cada vez que alguien envía una entrada a revisión. Con él puedes lanzar estas comprobaciones automáticamente (por ejemplo, en una función Lambda) y avisar a un revisor humano. Publicar y aprobar son acciones de IAM distintas: si las asignas a roles diferentes, un publicador no puede aprobar sus propias entradas.

In [ ]:
def revisar(entrada: dict) -> tuple[str, str]:
    """Política de TechStore para publicar en el catálogo. Devuelve (estado, motivo)."""
    metadatos = entrada.get("customMetadata", {})
    if metadatos.get("contiene_pii"):
        return "REJECTED", "Expone datos personales (PII): requiere una revisión de privacidad"
    if metadatos.get("clasificacion_datos") == "confidencial":
        return "REJECTED", "Los datos confidenciales no se publican en el catálogo general"
    if len(entrada.get("description", "")) < 60:
        return "REJECTED", "La descripción es demasiado breve para que otros equipos la encuentren"
    return "APPROVED", "Cumple la política de datos y la calidad de metadatos"


pendientes = control.list_registry_records(
    registryId=ID_REGISTRO, filters=[{"name": "status", "values": ["PENDING_APPROVAL"]}]
)["registryRecords"]

for pendiente in pendientes:
    entrada = control.get_registry_record(registryId=ID_REGISTRO, recordId=pendiente["recordId"])
    estado, motivo = revisar(entrada)
    control.update_registry_record_status(
        registryId=ID_REGISTRO, recordId=entrada["recordId"], status=estado, statusReason=motivo
    )
    print(f"{'✅' if estado == 'APPROVED' else '❌'} {entrada['name']}: {estado}. {motivo}")

tabla_entradas()

El motivo del rechazo queda guardado en la entrada (`statusReason`) para que el publicador sepa qué corregir antes de volver a enviarla.

Las entradas aprobadas aparecen al momento en el listado del plano de datos, pero el **índice de búsqueda** tarda unos segundos en actualizarse. Esperamos a que cada una se pueda encontrar por su nombre (`name` es uno de los campos por los que se puede filtrar):

In [ ]:
APROBADAS = {"techstore-pedidos", "techstore-soporte-tecnico", "politica-devoluciones"}


def encontrable(nombre: str) -> bool:
    """Indica si una entrada ya aparece en las búsquedas del plano de datos."""
    return bool(
        descubrimiento.search_discoverable_registry_records(
            registryIds=[ARN_REGISTRO], searchQuery=nombre, filters={"name": {"$eq": nombre}}, maxResults=1
        )["registryRecords"]
    )


def esperar_descubribles(nombres: set[str], limite: int = 180) -> None:
    """Espera a que las entradas indicadas aparezcan en las búsquedas del plano de datos."""
    inicio, pendientes = time.time(), set(nombres)
    while pendientes := {nombre for nombre in pendientes if not encontrable(nombre)}:
        if time.time() - inicio > limite:
            raise TimeoutError(f"Siguen sin aparecer: {sorted(pendientes)}")
        time.sleep(3)
    print(f"🔓 En las búsquedas tras {time.time() - inicio:.0f} s: {sorted(nombres)}")


esperar_descubribles(APROBADAS)

---

## Parte 4: El consumidor descubre

El plano de datos (`agent-registry`) ofrece tres operaciones, todas limitadas a entradas aprobadas:

| Operación | Para qué | Detalles |
|---|---|---|
| `SearchDiscoverableRegistryRecords` | Buscar con lenguaje natural | Búsqueda **híbrida** (semántica + palabras clave), hasta 20 resultados ordenados por relevancia, con filtros opcionales |
| `ListDiscoverableRegistryRecords` | Recorrer el catálogo | Paginado, filtra por tipo de entrada o de descriptor. Devuelve resúmenes sin descriptores |
| `BatchGetDiscoverableRegistryRecord` | Traer el detalle completo | Hasta 100 entradas por llamada |

### 4.1 Búsqueda en lenguaje natural

Ninguna de estas preguntas contiene el nombre de una entrada. Los resultados llegan ordenados por relevancia:

In [ ]:
def buscar(consulta: str, filtros: dict | None = None, maximo: int = 3) -> list[dict]:
    """Búsqueda híbrida (semántica + palabras clave) sobre las entradas aprobadas del registro."""
    parametros = {"registryIds": [ARN_REGISTRO], "searchQuery": consulta, "maxResults": maximo}
    if filtros:
        parametros["filters"] = filtros
    return descubrimiento.search_discoverable_registry_records(**parametros)["registryRecords"]


for consulta in [
    "¿dónde está mi paquete?",
    "mis auriculares no se emparejan con el móvil",
    "quiero que me devuelvan el dinero",
    "exportar la base de clientes a CSV",
]:
    nombres = [f"{e['name']} ({e['recordType']})" for e in buscar(consulta)]
    print(f"🔎 {consulta!r:48} → {nombres}")

La última búsqueda describe casi literalmente el exportador de Marketing y, aun así, **no aparece**: está rechazado, y para los consumidores no existe.

Con un catálogo tan pequeño, la búsqueda devuelve casi siempre las mismas entradas en distinto orden, y ese orden no siempre es el que esperarías. Con cientos de entradas, el orden marca la diferencia, porque un agente solo tomará los primeros resultados. Por eso importa escribir buenas descripciones y, como recomienda la documentación, **evaluar la búsqueda con consultas representativas** de tus usuarios. También por eso el orquestador de la Parte 6 lanza una búsqueda por cada capacidad que necesita.

### 4.2 Filtros sobre los metadatos

Los filtros admiten `$eq`, `$ne` e `$in` sobre los campos de la entrada (`name`, `recordType`, `recordVersion`) y sobre los metadatos personalizados con el prefijo `customMetadata.`, combinables con `$and` y `$or`:

In [ ]:
FILTROS = {
    "Solo servidores MCP": {"recordType": {"$eq": "MCP"}},
    "Solo del equipo de Soporte": {"customMetadata.equipo": {"$eq": "soporte"}},
    "Datos públicos o internos y sin PII": {
        "$and": [
            {"customMetadata.clasificacion_datos": {"$in": ["publico", "interno"]}},
            {"customMetadata.contiene_pii": {"$eq": False}},
        ]
    },
}

for descripcion, filtro in FILTROS.items():
    print(f"{descripcion:38} → {[e['name'] for e in buscar('TechStore', filtro, maximo=10)]}")

Más adelante, el orquestador de la Parte 6 aplicará siempre el último filtro: es la **política de gobierno** del asistente de cara al cliente, expresada como una consulta.

### 4.3 Qué contiene un resultado

Cada resultado incluye los metadatos de la entrada y sus **descriptores**: con ellos un agente sabe cómo conectarse. Lo que **no** incluye son credenciales. El registro describe dónde está cada capacidad, no cómo autenticarse en ella: esa parte la resuelve tu aplicación por un canal de confianza (por ejemplo, AgentCore Identity o AWS Secrets Manager).

In [ ]:
pedidos = buscar("consultar un pedido", {"recordType": {"$eq": "MCP"}}, maximo=1)[0]
servidor = json.loads(pedidos["descriptors"]["mcpServer"]["data"])
herramientas = json.loads(pedidos["descriptors"]["mcpServer"]["additionalData"]["tools"]["data"])["tools"]

print(f"Entrada:      {pedidos['name']} | versión {pedidos['recordVersion']} | {pedidos['status']}")
print(f"Metadatos:    {pedidos['customMetadata']}")
print(f"Conexión:     {servidor['remotes']}")
print(f"Herramientas: {[h['name'] for h in herramientas]}")

### 4.4 Valida antes de conectarte

Esta es una advertencia de la propia documentación: el registro valida los descriptores **contra el esquema**, pero no comprueba que las URL, los paquetes o los repositorios que citan sean de confianza. Según el modelo de responsabilidad compartida, **validarlos te corresponde a ti** antes de que un agente se conecte, descargue o instale nada.

El patrón más simple es una lista de dominios permitidos. En este taller solo permitimos servicios locales; en producción permitirías únicamente `https` y los dominios de tu empresa.

In [ ]:
from urllib.parse import urlparse

DOMINIOS_PERMITIDOS = {"127.0.0.1", "localhost"}


def endpoint_permitido(url: str) -> bool:
    """Valida el esquema y el dominio de una URL publicada en el registro antes de conectarse."""
    partes = urlparse(url)
    return partes.scheme in {"http", "https"} and partes.hostname in DOMINIOS_PERMITIDOS


for url in [servidor["remotes"][0]["url"], "https://exportador.marketing.example.com/mcp", "file:///etc/passwd"]:
    print(f"{'✅' if endpoint_permitido(url) else '⛔'} {url}")

---

## Parte 5: El registro como servidor MCP de un agente

Cada registro expone además un **endpoint MCP** con tres herramientas que reflejan el plano de datos: `search_discoverable_registry_records`, `list_discoverable_registry_records` y `batch_get_discoverable_registry_record`. Cualquier cliente MCP puede usarlo, desde un IDE como Kiro o Claude Code hasta un agente de Strands.

```
https://agent-registry.<región>.api.aws/registry/<registryId>/mcp
```

| Tipo de registro | Cómo se autentica el cliente MCP |
|---|---|
| IAM (el nuestro) | Peticiones firmadas con **SigV4**, servicio `agent-registry`. Permisos: `agent-registry:InvokeRegistryMcp` y `agent-registry:SearchDiscoverableRegistryRecords` |
| JWT | Cabecera `Authorization: Bearer <token>` de tu proveedor de identidad |

El `MCPClient` de Strands acepta `url=` y un `auth_provider` que puede ser cualquier `httpx.Auth`. Con `botocore` podemos escribir el firmante SigV4 en pocas líneas:

In [ ]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest


class SigV4HTTPXAuth(httpx.Auth):
    """Firma cada petición HTTP con SigV4 usando las credenciales de AWS de la sesión."""

    requires_request_body = True  # la firma incluye un hash del cuerpo

    def __init__(self, sesion: boto3.session.Session, servicio: str, region: str):
        self.sesion, self.servicio, self.region = sesion, servicio, region

    def auth_flow(self, request: httpx.Request):
        # "connection" es una cabecera de transporte (hop-by-hop): no se firma
        cabeceras = {k: v for k, v in request.headers.items() if k.lower() != "connection"}
        peticion_aws = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=cabeceras)
        credenciales = self.sesion.get_credentials().get_frozen_credentials()
        SigV4Auth(credenciales, self.servicio, self.region).add_auth(peticion_aws)
        request.headers.update(dict(peticion_aws.headers))
        yield request


URL_MCP_REGISTRO = f"https://agent-registry.{REGION}.api.aws/registry/{ID_REGISTRO}/mcp"


def cliente_registro() -> MCPClient:
    """Crea un cliente MCP para el endpoint del registro, autenticado con IAM."""
    return MCPClient(url=URL_MCP_REGISTRO, auth_provider=SigV4HTTPXAuth(sesion, "agent-registry", REGION))


with cliente_registro() as registro_mcp:
    for herramienta in registro_mcp.list_tools_sync():
        print(f"🛠️  {herramienta.tool_name}")

Ahora se lo damos a un agente. Es el caso de uso **en tiempo de desarrollo**: una desarrolladora va a construir un bot y, antes de escribir código, pregunta qué existe ya en la empresa. Pasamos el `MCPClient` directamente en `tools`: Strands abre y cierra la conexión por sí solo (*integración gestionada*).

In [ ]:
from strands import Agent

explorador = Agent(
    name="explorador",
    system_prompt="Eres un asistente para los desarrolladores de TechStore. Antes de construir algo nuevo, buscas "
    "en el catálogo de la empresa (AWS Agent Registry) si ya existe. Responde en español y de forma breve: "
    "nombre de la entrada, tipo, equipo responsable y cómo conectarse.",
    tools=[cliente_registro()],
    callback_handler=None,
)

respuesta = explorador(
    "Voy a construir un bot de atención al cliente. ¿Hay algo publicado para diagnosticar problemas "
    "con dispositivos? ¿Y para consultar pedidos?"
)
print("Herramientas del registro usadas:", list(respuesta.metrics.tool_metrics))
display(Markdown(str(respuesta)))
explorador.cleanup()

En un IDE, el mismo endpoint se configura en el `mcp.json`. Para registros con autorización IAM, la documentación recomienda [`mcp-proxy-for-aws`](https://github.com/aws/mcp-proxy-for-aws), que firma las peticiones por ti:

```json
{
  "mcpServers": {
    "catalogo-techstore": {
      "type": "stdio",
      "command": "uvx",
      "args": [
        "mcp-proxy-for-aws@latest",
        "https://agent-registry.<región>.api.aws/registry/<registryId>/mcp",
        "--service", "agent-registry",
        "--region", "<región>"
      ]
    }
  }
}
```

---

## Parte 6: Descubrir → validar → instanciar → ejecutar

Ahora el caso **en tiempo de ejecución**. El orquestador de TechStore **no tiene ninguna herramienta cableada**: en cada petición busca en el registro lo que necesita, lo conecta y lo usa.

```
 petición del cliente
        │
        ▼
┌──────────────┐  consultas  ┌───────────────────┐
│ orquestador  │ ──────────▶ │ AWS Agent Registry│  solo entradas APPROVED
└──────────────┘             └───────────────────┘  que cumplen el filtro de gobierno
        │                            │ descriptores (sin credenciales)
        ▼                            ▼
 descubrir_y_ejecutar ──▶ validar endpoint ──▶ MCP   → MCPClient
                                               AGENT → A2AAgent (como herramienta)
                                               SKILL → Skill.from_content
                                                    │
                                                    ▼
                            agente efímero con SOLO esas capacidades → respuesta
```

| Paso | Qué hace | Pieza |
|---|---|---|
| Descubrir | Una búsqueda por cada capacidad necesaria, con el filtro de gobierno | `search_discoverable_registry_records` |
| Validar | Solo se conecta a dominios permitidos | `endpoint_permitido` |
| Instanciar | Convierte cada descriptor en algo que Strands sabe usar | `MCPClient`, `A2AAgent`, `Skill` |
| Ejecutar | Un agente nuevo con solo lo descubierto; al terminar se cierran sus conexiones | `Agent` + `cleanup()` |

Primero, la función que traduce una entrada en una capacidad de Strands. Un `A2AAgent` no es una herramienta, así que lo envolvemos en una cuyo nombre y descripción salen de su Agent Card.

In [ ]:
from datetime import date

from strands import AgentSkills, Skill, tool
from strands.agent.a2a_agent import A2AAgent

# Política del asistente de cara al cliente: solo capacidades públicas o internas y sin PII
FILTRO_GOBIERNO = {
    "$and": [
        {"customMetadata.clasificacion_datos": {"$in": ["publico", "interno"]}},
        {"customMetadata.contiene_pii": {"$eq": False}},
    ]
}


def herramienta_a2a(tarjeta: dict):
    """Envuelve un agente remoto A2A en una herramienta con el nombre y la descripción de su Agent Card."""
    remoto = A2AAgent(endpoint=tarjeta["url"], timeout=120)

    @tool(name=f"agente_{tarjeta['name']}", description=tarjeta["description"])
    def llamar_agente(mensaje: str) -> str:
        """Envía un mensaje al agente remoto.

        Args:
            mensaje: Lo que necesitas del agente, con todo el contexto
        """
        return str(remoto(mensaje))

    return llamar_agente


def instanciar(entrada: dict):
    """Convierte una entrada del registro en una capacidad de Strands, o None si su endpoint no está permitido."""
    descriptores = entrada["descriptors"]
    if entrada["recordType"] == "MCP":
        remotos = json.loads(descriptores["mcpServer"]["data"]).get("remotes", [])
        url = remotos[0]["url"] if remotos else ""
        return MCPClient(url=url) if endpoint_permitido(url) else None
    if entrada["recordType"] == "AGENT":
        tarjeta = json.loads(descriptores["a2aAgentCard"]["data"])
        return herramienta_a2a(tarjeta) if endpoint_permitido(tarjeta["url"]) else None
    if entrada["recordType"] == "SKILL":
        return Skill.from_content(descriptores["agentSkillsDefinition"]["additionalData"]["skillMd"]["data"])
    return None

Y ahora la herramienta del orquestador. El modelo decide **qué** buscar (una consulta breve por capacidad) y la herramienta hace el resto. Lanzar varias consultas y quitar los duplicados mejora la cobertura cuando una petición necesita más de una capacidad.

Como en el Módulo 9, creamos un orquestador nuevo para cada petición, para que no arrastre el historial de la anterior.

In [ ]:
@tool
def descubrir_y_ejecutar(consultas: list[str], peticion: str) -> str:
    """Busca en el catálogo de TechStore las capacidades necesarias, las conecta y resuelve la petición con ellas.

    Args:
        consultas: Una búsqueda breve por cada capacidad que necesitas, por ejemplo "consultar el estado de un pedido"
        peticion: La petición completa del cliente, con todos sus datos
    """
    # 1. Descubrir: una búsqueda por capacidad, con el filtro de gobierno y sin duplicados
    encontradas = {}
    for consulta in consultas:
        for entrada in buscar(consulta, FILTRO_GOBIERNO, maximo=2):
            encontradas[entrada["recordId"]] = entrada
    print(f"   🔎 {consultas} → {[e['name'] for e in encontradas.values()]}")

    # 2 y 3. Validar e instanciar
    herramientas, skills = [], []
    for entrada in encontradas.values():
        capacidad = instanciar(entrada)
        if capacidad is None:
            print(f"   ⛔ {entrada['name']}: endpoint no permitido, se descarta")
        elif isinstance(capacidad, Skill):
            skills.append(capacidad)
        else:
            herramientas.append(capacidad)

    # 4. Ejecutar con un agente efímero que solo tiene lo descubierto
    agente = Agent(
        name="asistente_efimero",
        system_prompt=f"Eres el asistente de atención al cliente de TechStore. Hoy es {date.today().isoformat()}. "
        "Usa solo las herramientas y skills disponibles. No inventes procedimientos, soluciones técnicas ni datos "
        "de contacto: si ninguna capacidad cubre la petición, dilo con claridad y ofrece pasarla a un agente "
        "humano. Responde en español, en menos de 120 palabras.",
        tools=herramientas,
        plugins=[AgentSkills(skills=skills)] if skills else [],
        callback_handler=None,
    )
    try:
        resultado = agente(peticion)
        print(f"   🧰 Capacidades: {agente.tool_names} | usadas: {list(resultado.metrics.tool_metrics)}")
        return str(resultado)
    finally:
        agente.cleanup()  # cierra las conexiones MCP de este turno


def atender(peticion: str) -> None:
    """Crea un orquestador nuevo y le pasa la petición del cliente."""
    orquestador = Agent(
        name="orquestador",
        system_prompt="Eres el orquestador de TechStore y no tienes capacidades propias. Para cada petición del "
        "cliente, decide qué capacidades necesitas y llama UNA sola vez a descubrir_y_ejecutar, con una búsqueda "
        "breve por capacidad. Devuelve al cliente la respuesta que obtengas, sin añadir nada.",
        tools=[descubrir_y_ejecutar],
        callback_handler=None,
    )
    print(f"👤 {peticion}")
    display(Markdown(f"🤖 {orquestador(peticion)}"))

**Caso 1:** una pregunta sobre un envío. Debería bastar el servidor MCP de Pedidos.

In [ ]:
atender("¿Dónde está mi pedido ORD-9034? ¿Cuándo me llega?")

**Caso 2:** una avería. Necesita el agente A2A de Soporte y, posiblemente, los datos del pedido.

In [ ]:
atender("Mis auriculares del pedido ORD-5521 no se emparejan con el móvil. ¿Qué hago?")

**Caso 3:** una devolución. La skill aporta la política y el servidor MCP, los datos del pedido.

In [ ]:
atender("Quiero devolver el teclado del pedido ORD-7788. ¿Me devolvéis el dinero?")

En las líneas 🔎 y 🧰 ves qué encontró cada búsqueda y qué capacidades recibió el agente efímero. Ni el orquestador ni `descubrir_y_ejecutar` importan el servidor de Pedidos, el agente de Soporte o la skill: **todo llega del registro**. Si mañana el equipo de Pagos publica un servidor MCP y el curador lo aprueba, el orquestador podrá usarlo sin cambiar una línea de código.

---

## Parte 7: Gobierno en acción

El catálogo no es estático: los equipos actualizan sus entradas y retiran las que ya no mantienen. Veamos cómo protege el registro a los consumidores en los dos casos.

### 7.1 Un cambio sin aprobar no llega a los agentes

Con prisa por una campaña, el equipo de Atención al cliente amplía el plazo de devolución de 30 a **90 días** directamente en su skill, sin consultar a Finanzas. Al editar una entrada aprobada, esta vuelve a `DRAFT` y tiene que pasar de nuevo por revisión.

La pregunta clave es: **¿qué ven los agentes mientras tanto?**

In [ ]:
ID_SKILL = entradas["politica-devoluciones"]
skill_90_dias = skill_md.replace("30 días o menos", "90 días o menos").replace("Más de 30 días", "Más de 90 días")

# En la API de actualización, cada campo opcional va envuelto en "optionalValue"
skill_md_nuevo = {"skillMd": {"optionalValue": {"data": {"optionalValue": skill_90_dias}}}}
definicion_nueva = {"additionalData": {"optionalValue": skill_md_nuevo}}
control.update_registry_record(
    registryId=ID_REGISTRO,
    recordId=ID_SKILL,
    recordVersion="1.1.0",
    descriptors={"optionalValue": {"agentSkillsDefinition": {"optionalValue": definicion_nueva}}},
)
print("Estado tras editar:", esperar_entrada(ID_SKILL))


def comparar_planos() -> None:
    """Compara la versión de la skill en el plano de control con la que reciben los agentes."""
    borrador = control.get_registry_record(registryId=ID_REGISTRO, recordId=ID_SKILL)
    publicada = descubrimiento.batch_get_discoverable_registry_record(
        entries=[{"registryId": ID_REGISTRO, "recordIds": [ID_SKILL]}]
    )["registryRecords"][0]
    for plano, e in [("Plano de control", borrador), ("Plano de datos (agentes)", publicada)]:
        md_skill = e["descriptors"]["agentSkillsDefinition"]["additionalData"]["skillMd"]["data"]
        plazo = "90 días" if "90 días o menos" in md_skill else "30 días"
        print(f"{plano:26} → versión {e['recordVersion']} | {e['status']:16} | plazo de devolución: {plazo}")


comparar_planos()

El borrador con 90 días existe, pero los agentes **siguen recibiendo la versión aprobada**. Lo comprobamos con un pedido entregado hace 45 días: con la política aprobada ya no admite devolución.

In [ ]:
atender("Quiero devolver el monitor del pedido ORD-3120. ¿Puedo?")

El equipo envía el cambio a revisión y el curador lo rechaza: los cambios en la política de reembolsos necesitan el visto bueno de Finanzas.

In [ ]:
envio = control.submit_registry_record_for_approval(registryId=ID_REGISTRO, recordId=ID_SKILL)
print("Envío a revisión:", envio["status"])
control.update_registry_record_status(
    registryId=ID_REGISTRO,
    recordId=ID_SKILL,
    status="REJECTED",
    statusReason="Los cambios en la política de reembolsos requieren la aprobación de Finanzas",
)
comparar_planos()

Aunque la nueva versión está rechazada, los agentes siguen usando la última aprobada (1.0.0). **Una edición nunca cambia en silencio lo que descubren los consumidores**: cada versión pasa por la misma revisión que la primera.

### 7.2 Retirar una capacidad

El equipo de Soporte va a sustituir su agente por una versión nueva y el curador **retira** (`DEPRECATED`) la actual. Es un estado final: la entrada deja de ser descubrible y no puede volver a aprobarse.

In [ ]:
control.update_registry_record_status(
    registryId=ID_REGISTRO,
    recordId=entradas["techstore-soporte-tecnico"],
    status="DEPRECATED",
    statusReason="Sustituido por la versión 2 del agente de soporte",
)
print("Búsqueda de soporte técnico →", [e["name"] for e in buscar("diagnosticar averías de dispositivos")])
tabla_entradas()

En la tabla, `politica-devoluciones` figura como `REJECTED` porque el plano de control muestra su última versión (la 1.1.0 rechazada); los agentes siguen recibiendo la 1.0.0 aprobada.

Repetimos el caso 2. **No hemos tocado el código del orquestador**, pero en la línea 🔎 verás que la búsqueda ya no devuelve al especialista de soporte:

In [ ]:
atender("Mis auriculares del pedido ORD-5521 no se emparejan con el móvil. ¿Qué hago?")

El servicio A2A sigue funcionando en su puerto, pero **el catálogo ya no lo ofrece** y ningún agente que descubra sus capacidades a través del registro lo usará. Así un curador puede apagar el uso de una capacidad problemática en toda la organización con una sola llamada.

---

## Parte 8: Limpieza

Borramos las entradas, después el registro (no se puede borrar mientras tenga entradas) y, por último, detenemos los servicios locales.

In [ ]:
for entrada in control.list_registry_records(registryId=ID_REGISTRO)["registryRecords"]:
    control.delete_registry_record(registryId=ID_REGISTRO, recordId=entrada["recordId"])
    print(f"🗑️  {entrada['name']}")

# El registro solo se puede borrar cuando ya no tiene entradas
while control.list_registry_records(registryId=ID_REGISTRO)["registryRecords"]:
    time.sleep(3)
print("Registro:", control.delete_registry(registryId=ID_REGISTRO)["status"])

for proceso, log in [(servicio_mcp, log_mcp), (servicio_a2a, log_a2a)]:
    proceso.terminate()
    proceso.wait()
    log.close()
print("🛑 Servicios locales detenidos")

---

## Hacia producción

| Tema | Qué ofrece AWS Agent Registry |
|---|---|
| Autorización | IAM (SigV4) o JWT de tu proveedor de identidad, con creación rápida en Amazon Cognito. Se elige al crear el registro y no se puede cambiar |
| Separación de funciones | Acciones de IAM distintas para publicar (`CreateRegistryRecord`, `SubmitRegistryRecordForApproval`), aprobar (`UpdateRegistryRecordStatus`) y descubrir (`SearchDiscoverableRegistryRecords`, `InvokeRegistryMcp`) |
| Flujo de aprobación | Eventos de Amazon EventBridge cuando una entrada se envía a revisión, para conectar comprobaciones automáticas y revisiones humanas |
| Auditoría | AWS CloudTrail registra todas las llamadas a la API del registro |
| Escala | Registros compartidos entre cuentas con AWS RAM, registros para toda la organización y **detección automática** de agentes en AgentCore Runtime y AgentCore Gateway |
| Infraestructura como código | AWS CloudFormation (`AWS::AgentRegistry::Registry`), Terraform y AWS CDK |
| Sincronización | Una entrada puede leer sus metadatos de un servidor MCP o A2A accesible por URL (`source.fromUrl`) y mantenerlos al día |
| Consumo | Además de por API y MCP, el registro se integra con Amazon Quick y con Kiro |
| Credenciales | El registro no guarda ni devuelve credenciales de invocación: gestiónalas con AgentCore Identity o AWS Secrets Manager |
| Validación | Comprueba URL, paquetes y repositorios con listas de permitidos antes de usarlos |
| Espacio de nombres | Usa los clientes `agent-registry-control` y `agent-registry`. El espacio de nombres de la vista previa (`bedrock-agentcore`) deja de estar disponible el 30 de octubre de 2026 |

---

## Resumen

| Lo que hiciste | Pieza |
|---|---|
| Creaste un registro con aprobación manual y metadatos obligatorios | `create_registry` + `approvalConfiguration` + `customMetadataSchemaConfiguration` |
| Publicaste un servidor MCP, un agente A2A y una skill | `create_registry_record` con descriptores `mcpServer`, `a2aAgentCard` y `agentSkillsDefinition` |
| Bloqueaste una entrada sin metadatos y rechazaste otra con PII | Validación del esquema al enviar + `update_registry_record_status` |
| Buscaste con lenguaje natural y con filtros | `search_discoverable_registry_records` + `filters` sobre `customMetadata` |
| Validaste los endpoints antes de conectarte | Lista de dominios permitidos |
| Conectaste un agente de Strands al endpoint MCP del registro | `MCPClient(url=..., auth_provider=SigV4HTTPXAuth(...))` |
| Construiste un orquestador sin herramientas cableadas | Descubrir → validar → instanciar (`MCPClient`, `A2AAgent`, `Skill`) → ejecutar |
| Comprobaste que una edición sin aprobar no llega a los agentes y retiraste una capacidad | `update_registry_record`, `batch_get_discoverable_registry_record` y `DEPRECATED` |

**La idea central:** con AWS Agent Registry, lo que un agente **puede** usar deja de decidirse en su código y pasa a decidirse en un catálogo gobernado. Los equipos publican a su ritmo, el curador decide qué está aprobado y los agentes lo descubren en tiempo de ejecución. Y como solo lo aprobado es descubrible, aprobar, rechazar o retirar una entrada cambia al momento lo que pueden usar todos los agentes de la organización.